## 5.5 classification part B (Pokemon, `is_legendary`)

Part B trains the remaining 5 algorithms: Random Forest, AdaBoost, Gradient Boosting, Bagging (Decision Tree base) and MLP. They use the **same** stratified split and the **same** leak-free preprocessing pipeline as Part A, so all 10 models are directly comparable.

> Paste these cells at the end of the Review 1 notebook. They reuse `Xp_train`, `Xp_test`, `yp_train`, `yp_test`, `get_poke_preprocessor`, `poke_cls_results`, `skf_p` and `SEED`.

In [ ]:
from IPython.display import display
from sklearn.base import clone
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier, GradientBoostingClassifier, BaggingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import roc_auc_score, roc_curve

### 5.5.1 train the Part B models

In [ ]:
poke_classifiers_b = {
    "Random Forest": RandomForestClassifier(random_state=SEED),
    "AdaBoost": AdaBoostClassifier(random_state=SEED),
    "Gradient Boosting": GradientBoostingClassifier(random_state=SEED),
    # note: older sklearn (<1.2) uses base_estimator= instead of estimator=
    "Bagging (Decision Tree)": BaggingClassifier(
        estimator=DecisionTreeClassifier(random_state=SEED), n_estimators=50, random_state=SEED),
    "MLP Classifier": MLPClassifier(hidden_layer_sizes=(64, 32), activation="relu",
                                    max_iter=1000, random_state=SEED),
}

poke_cls_results_b = {}
for name, clf in poke_classifiers_b.items():
    pipe_b = Pipeline(steps=[("prep", get_poke_preprocessor()), ("clf", clf)])
    pipe_b.fit(Xp_train, yp_train)
    poke_cls_results_b[name] = {"pipe": pipe_b, "preds": pipe_b.predict(Xp_test)}
    print(f"trained: {name}")

### 5.5.2 consolidated comparison table (all 10 algorithms)

Every model, Part A and Part B, is scored with the same function on the same test set. Precision, recall and F1 are **weighted** as the guidelines require. Because legendaries are rare, an extra `F1 (Legendary)` column shows how well the minority class is handled.

In [ ]:
def get_scores(pipe, X):
    # probability of the positive class; falls back to decision_function (e.g. SVC without probability=True)
    if hasattr(pipe, "predict_proba"):
        return pipe.predict_proba(X)[:, 1]
    return pipe.decision_function(X)

def evaluate_model(name, part, pipe, X, y):
    preds = pipe.predict(X)
    return {
        "Model": name,
        "Part": part,
        "Accuracy": accuracy_score(y, preds),
        "Precision": precision_score(y, preds, average="weighted", zero_division=0),
        "Recall": recall_score(y, preds, average="weighted", zero_division=0),
        "F1-Score": f1_score(y, preds, average="weighted", zero_division=0),
        "ROC-AUC": roc_auc_score(y, get_scores(pipe, X)),
        "F1 (Legendary)": f1_score(y, preds, pos_label=1, zero_division=0),
    }

all_cls = {**poke_cls_results, **poke_cls_results_b}
part_of = {n: "A" for n in poke_cls_results} | {n: "B" for n in poke_cls_results_b}

final_cls_df = (pd.DataFrame([evaluate_model(n, part_of[n], i["pipe"], Xp_test, yp_test)
                              for n, i in all_cls.items()])
                .sort_values(by=["F1-Score", "ROC-AUC"], ascending=False)
                .reset_index(drop=True))
final_cls_df.index += 1   # rank starts at 1
display(final_cls_df.round(4))

### 5.5.3 ROC curves (all 10 models)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
palette = sns.color_palette("tab10", len(all_cls))
for (name, info), color in zip(all_cls.items(), palette):
    scores = get_scores(info["pipe"], Xp_test)
    fpr, tpr, _ = roc_curve(yp_test, scores)
    ax.plot(fpr, tpr, color=color, lw=1.8, label=f"{name} (AUC={roc_auc_score(yp_test, scores):.3f})")
ax.plot([0, 1], [0, 1], "k--", label="chance")
ax.set_xlabel("false positive rate")
ax.set_ylabel("true positive rate")
ax.set_title("ROC curves: all 10 classifiers (is_legendary)")
ax.legend(fontsize=8, loc="lower right")
plt.tight_layout()
plt.show()

### 5.5.4 confusion matrices (Part B)

In [ ]:
fig, axes = plt.subplots(1, len(poke_cls_results_b), figsize=(4 * len(poke_cls_results_b), 4))
for ax, (name, info) in zip(axes, poke_cls_results_b.items()):
    cm = confusion_matrix(yp_test, info["preds"])
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
                xticklabels=["Not Legendary", "Legendary"], yticklabels=["Not Legendary", "Legendary"])
    ax.set_title(name, fontsize=10)
    ax.set_xlabel("predicted")
    ax.set_ylabel("actual")
plt.tight_layout()
plt.show()

for name, info in poke_cls_results_b.items():
    print(f"\n--- classification report ({name}) ---")
    print(classification_report(yp_test, info["preds"], target_names=["Not Legendary", "Legendary"]))

### 5.5.5 feature importance (Random Forest) and decision tree visualisation

In [ ]:
def clean_names(pipe):
    return [n.split("__", 1)[1] for n in pipe.named_steps["prep"].get_feature_names_out()]

# random forest feature importance
rf_pipe = poke_cls_results_b["Random Forest"]["pipe"]
rf_imp = (pd.DataFrame({"Feature": clean_names(rf_pipe),
                        "Importance": rf_pipe.named_steps["clf"].feature_importances_})
          .sort_values("Importance", ascending=False).head(10))

plt.figure(figsize=(8, 4))
plt.barh(rf_imp["Feature"][::-1], rf_imp["Importance"][::-1], color="teal")
plt.title("top 10 feature importances (Random Forest)")
plt.xlabel("importance")
plt.ylabel("feature")
plt.tight_layout()
plt.show()

# decision tree from Part A (top 3 levels only, so it stays readable)
dt_pipe = poke_cls_results["Decision Tree Classifier"]["pipe"]
plt.figure(figsize=(18, 8))
plot_tree(dt_pipe.named_steps["clf"], max_depth=3, feature_names=clean_names(dt_pipe),
          class_names=["Not Legendary", "Legendary"], filled=True, fontsize=8)
plt.title("decision tree (first 3 levels)")
plt.show()

**feature importance / tree observations:** *(write your own interpretation here: which stats drive the legendary split and why)*

### 5.5.6 cross-validation of the two best models

5-fold stratified CV on the training set only, so the test set stays untouched.

In [ ]:
top2 = final_cls_df["Model"].head(2).tolist()
cv_rows = []
for name in top2:
    pipe_cv = all_cls[name]["pipe"]   # cross_val_score clones it, so the fitted model is not affected
    f1_cv = cross_val_score(pipe_cv, Xp_train, yp_train, cv=skf_p, scoring="f1_weighted")
    auc_cv = cross_val_score(pipe_cv, Xp_train, yp_train, cv=skf_p, scoring="roc_auc")
    cv_rows.append({"Model": name,
                    "CV F1 (weighted) mean": f1_cv.mean(), "CV F1 std": f1_cv.std(),
                    "CV ROC-AUC mean": auc_cv.mean(), "CV ROC-AUC std": auc_cv.std()})
display(pd.DataFrame(cv_rows).round(4))

### 5.5.7 final model selection and hyperparameter tuning

The best model from the table above is tuned with `GridSearchCV` on the training set (scoring = weighted F1). Set `FINAL_MODEL` manually if you want to tune a different model.

In [ ]:
param_grids = {
    "Logistic Regression": {"clf__C": [0.01, 0.1, 1, 10, 100]},
    "KNN Classifier": {"clf__n_neighbors": [3, 5, 7, 9, 11], "clf__weights": ["uniform", "distance"]},
    "Naive Bayes": {"clf__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]},
    "Decision Tree Classifier": {"clf__max_depth": [3, 5, 7, 10, None], "clf__min_samples_split": [2, 5, 10]},
    "SVC": {"clf__C": [0.1, 1, 10, 100], "clf__kernel": ["linear", "rbf"], "clf__gamma": ["scale", "auto"]},
    "Random Forest": {"clf__n_estimators": [100, 200, 300], "clf__max_depth": [None, 10, 20],
                      "clf__min_samples_split": [2, 5]},
    "AdaBoost": {"clf__n_estimators": [50, 100, 200], "clf__learning_rate": [0.01, 0.1, 0.5, 1.0]},
    "Gradient Boosting": {"clf__n_estimators": [100, 200], "clf__learning_rate": [0.05, 0.1, 0.2],
                          "clf__max_depth": [2, 3, 4]},
    "Bagging (Decision Tree)": {"clf__n_estimators": [10, 50, 100], "clf__max_samples": [0.5, 0.8, 1.0],
                                "clf__max_features": [0.8, 1.0]},
    "MLP Classifier": {"clf__hidden_layer_sizes": [(32,), (64, 32), (128, 64)],
                       "clf__activation": ["relu", "tanh"], "clf__alpha": [1e-4, 1e-3, 1e-2]},
}

FINAL_MODEL = None   # e.g. "Random Forest"; None = best model from the table
final_name = FINAL_MODEL or final_cls_df.loc[1, "Model"]
print(f"tuning: {final_name}")

grid_cls = GridSearchCV(clone(all_cls[final_name]["pipe"]), param_grids[final_name],
                        cv=skf_p, scoring="f1_weighted", n_jobs=-1)
grid_cls.fit(Xp_train, yp_train)
best_cls = grid_cls.best_estimator_

print("best params:", grid_cls.best_params_)
print(f"best CV weighted F1: {grid_cls.best_score_:.4f}")

before = evaluate_model(f"{final_name} (baseline)", part_of[final_name], all_cls[final_name]["pipe"], Xp_test, yp_test)
after = evaluate_model(f"{final_name} (tuned)", part_of[final_name], best_cls, Xp_test, yp_test)
tune_df = pd.DataFrame([before, after]).drop(columns="Part").set_index("Model")
tune_df.loc["delta"] = tune_df.iloc[1] - tune_df.iloc[0]
display(tune_df.round(4))

### 5.5.8 final model justification

*(Write this in your own words. Suggested points:)*
- which model won and on which metrics (F1, ROC-AUC, recall on legendaries)
- why that model suits this data (small dataset, class imbalance, correlated stats, non-linear boundary)
- how much tuning improved it, and whether the gain is meaningful or within noise (compare with the CV std)
- trade-offs: training time, interpretability, overfitting risk

In [ ]:
# optional: save the tuned model (models/ folder in the repo); also useful for the Streamlit/Gradio bonus
import joblib
os.makedirs("models", exist_ok=True)
joblib.dump(best_cls, "models/pokemon_legendary_classifier.pkl")
print("saved models/pokemon_legendary_classifier.pkl")